In [14]:
# Run this cell FIRST in Day 13 notebook
import json

# Force correct threshold
threshold_config = {
    'final_threshold': 0.5,
    'best_f1_threshold': 0.5,
    'high_precision_threshold': 0.98,
    'note': 'Using 0.5 for high recall — '
            'catching more fraud is priority'
}

with open('../models/threshold_config.json', 'w') as f:
    json.dump(threshold_config, f, indent=2)

print("✅ Threshold fixed!")
print(f"final_threshold = 0.5 (high recall) ✅")
print(f"0.98 saved as high_precision option only")

✅ Threshold fixed!
final_threshold = 0.5 (high recall) ✅
0.98 saved as high_precision option only


In [15]:
# Cell 1 — Check what we have so far
import os
import pickle
import json
import pandas as pd
import numpy as np

print("=== CHECKING EXISTING MODEL FILES ===\n")

files_to_check = [
    '../models/xgb_fraud_model.pkl',
    '../models/shap_explainer.pkl',
    '../models/scaler_amount.pkl',
    '../models/scaler_hour.pkl',
    '../models/threshold_config.json',
    '../models/baseline_results.json',
    '../models/xgb_results.json',
]

for filepath in files_to_check:
    exists = os.path.exists(filepath)
    size = os.path.getsize(filepath) / 1024 \
        if exists else 0
    status = "✅" if exists else "❌ MISSING"
    print(f"{status} {filepath:<45} "
          f"{size:.1f} KB" if exists else
          f"{status} {filepath}")

=== CHECKING EXISTING MODEL FILES ===

✅ ../models/xgb_fraud_model.pkl                 826.1 KB
✅ ../models/shap_explainer.pkl                  2693.0 KB
✅ ../models/scaler_amount.pkl                   0.6 KB
✅ ../models/scaler_hour.pkl                     0.6 KB
✅ ../models/threshold_config.json               0.2 KB
✅ ../models/baseline_results.json               0.9 KB
✅ ../models/xgb_results.json                    1.4 KB


In [16]:
# Cell 2 — Load everything
print("Loading all components...\n")

# Model
with open('../models/xgb_fraud_model.pkl', 'rb') as f:
    model = pickle.load(f)
print("✅ XGBoost model loaded")

# SHAP explainer
with open('../models/shap_explainer.pkl', 'rb') as f:
    explainer = pickle.load(f)
print("✅ SHAP explainer loaded")

# Scalers
with open('../models/scaler_amount.pkl', 'rb') as f:
    scaler_amount = pickle.load(f)
with open('../models/scaler_hour.pkl', 'rb') as f:
    scaler_hour = pickle.load(f)
print("✅ Scalers loaded")

# Threshold config
with open('../models/threshold_config.json', 'r') as f:
    threshold_config = json.load(f)
print("✅ Threshold config loaded")

# Test data for verification
X_test = pd.read_csv('../data/splits/X_test.csv')
y_test = pd.read_csv(
    '../data/splits/y_test.csv'
).squeeze()

print(f"\n=== LOADED COMPONENTS ===")
print(f"Model type:      {type(model).__name__}")
print(f"Model features:  {model.n_features_in_}")
print(f"Threshold:       {threshold_config['final_threshold']}")
print(f"Base SHAP value: {explainer.expected_value:.4f}")

Loading all components...

✅ XGBoost model loaded
✅ SHAP explainer loaded
✅ Scalers loaded
✅ Threshold config loaded

=== LOADED COMPONENTS ===
Model type:      XGBClassifier
Model features:  32
Threshold:       0.5
Base SHAP value: 2.6191


In [17]:
# Cell 3 — Quick verification
from sklearn.metrics import roc_auc_score, f1_score

y_prob = model.predict_proba(X_test)[:, 1]
threshold = threshold_config['final_threshold']
y_pred = (y_prob >= threshold).astype(int)

auc = roc_auc_score(y_test, y_prob)
f1 = f1_score(y_test, y_pred)

print("=== MODEL VERIFICATION ===\n")
print(f"AUC-ROC: {auc:.4f}  ✅" if auc > 0.95
      else f"AUC-ROC: {auc:.4f}  ⚠️ Check model!")
print(f"F1:      {f1:.4f}  ✅" if f1 > 0.80
      else f"F1:      {f1:.4f}  ⚠️ Check model!")
print(f"\nModel is working correctly! ✅")

=== MODEL VERIFICATION ===

AUC-ROC: 0.9817  ✅
F1:      0.8454  ✅

Model is working correctly! ✅


In [18]:
# Cell 4 — Create complete pipeline
print("Building complete pipeline...")

def preprocess_transaction(raw_transaction):
    """
    Preprocesses a raw transaction dict
    into model-ready format.

    Input: dict with Amount, Time, V1-V28
    Output: pd.DataFrame ready for model
    """
    df = pd.DataFrame([raw_transaction])

    # Feature engineering (same as Day 4)
    df['hour'] = (df['Time'] // 3600) % 24
    df['amount_log'] = np.log(df['Amount'] + 1)
    df['amount_scaled'] = scaler_amount.transform(
        df[['Amount']]
    )
    df['hour_scaled'] = scaler_hour.transform(
        df[['hour']]
    )

    # Drop original columns
    df.drop(['Time', 'Amount'], axis=1, inplace=True)

    # Ensure correct column order
    expected_cols = X_test.columns.tolist()
    df = df[expected_cols]

    return df


def predict_fraud(raw_transaction,
                  return_explanation=True):
    """
    Complete fraud prediction pipeline.

    Input:  raw transaction dict
    Output: prediction + explanation

    Steps:
    1. Preprocess transaction
    2. Get fraud probability
    3. Apply threshold
    4. Generate SHAP explanation
    5. Return complete result
    """

    # Step 1: Preprocess
    processed = preprocess_transaction(
        raw_transaction
    )

    # Step 2: Get probability
    prob = model.predict_proba(processed)[0][1]

    # Step 3: Apply threshold
    threshold = threshold_config['final_threshold']
    is_fraud = bool(prob >= threshold)

    # Step 4: Risk level
    if prob >= 0.7:
        risk = "HIGH"
    elif prob >= 0.4:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    result = {
        'fraud_probability': round(float(prob), 4),
        'is_fraud': is_fraud,
        'risk_level': risk,
        'threshold_used': threshold
    }

    # Step 5: SHAP explanation (optional)
    if return_explanation:
        shap_vals = explainer.shap_values(processed)

        contributions = pd.DataFrame({
            'feature': processed.columns,
            'feature_value': processed.values[0],
            'shap_value': shap_vals[0]
        }).sort_values(
            'shap_value', key=abs, ascending=False
        )

        # Top fraud signals
        fraud_signals = contributions[
            contributions['shap_value'] > 0
        ].head(3)

        # Top legit signals
        legit_signals = contributions[
            contributions['shap_value'] < 0
        ].head(3)

        result['explanation'] = {
            'base_rate': round(
                float(explainer.expected_value), 4
            ),
            'top_fraud_signals': [
                {
                    'feature': row['feature'],
                    'value': round(
                        float(row['feature_value']), 4
                    ),
                    'impact': round(
                        float(row['shap_value']), 4
                    )
                }
                for _, row in fraud_signals.iterrows()
            ],
            'top_legit_signals': [
                {
                    'feature': row['feature'],
                    'value': round(
                        float(row['feature_value']), 4
                    ),
                    'impact': round(
                        float(row['shap_value']), 4
                    )
                }
                for _, row in legit_signals.iterrows()
            ]
        }

    return result


print("✅ Pipeline functions created!")
print("\nFunctions available:")
print("  preprocess_transaction() → prepares raw data")
print("  predict_fraud()          → full prediction")

Building complete pipeline...
✅ Pipeline functions created!

Functions available:
  preprocess_transaction() → prepares raw data
  predict_fraud()          → full prediction


In [19]:
# Cell 5 — Test with real transactions

# Test 1: Known fraud transaction
fraud_idx = np.where(y_test == 1)[0][0]
fraud_raw = X_test.iloc[fraud_idx].to_dict()

# Add back Time and Amount for preprocessing
# (in real API these come from raw transaction)
fraud_raw['Time'] = 50000
fraud_raw['Amount'] = 150.0

print("=== TEST 1: FRAUD TRANSACTION ===\n")
result_fraud = predict_fraud(fraud_raw)

print(f"Fraud Probability: {result_fraud['fraud_probability']}")
print(f"Is Fraud:          {result_fraud['is_fraud']}")
print(f"Risk Level:        {result_fraud['risk_level']}")
print(f"\nTop Fraud Signals:")
for sig in result_fraud['explanation']['top_fraud_signals']:
    print(f"  {sig['feature']}: "
          f"value={sig['value']:.3f}, "
          f"impact=+{sig['impact']:.4f}")
print(f"\nTop Legit Signals:")
for sig in result_fraud['explanation']['top_legit_signals']:
    print(f"  {sig['feature']}: "
          f"value={sig['value']:.3f}, "
          f"impact={sig['impact']:.4f}")

=== TEST 1: FRAUD TRANSACTION ===

Fraud Probability: 0.9999
Is Fraud:          True
Risk Level:        HIGH

Top Fraud Signals:
  V14: value=-6.174, impact=+4.4111
  V10: value=-4.881, impact=+1.6336
  V12: value=-4.686, impact=+1.2009

Top Legit Signals:
  V6: value=-0.948, impact=-0.5440
  V8: value=1.167, impact=-0.5095
  V19: value=1.716, impact=-0.3376


In [20]:
# Cell 6 — Test 2: Known legit transaction
legit_idx = np.where(y_test == 0)[0][0]
legit_raw = X_test.iloc[legit_idx].to_dict()
legit_raw['Time'] = 50000
legit_raw['Amount'] = 50.0

print("=== TEST 2: LEGIT TRANSACTION ===\n")
result_legit = predict_fraud(legit_raw)

print(f"Fraud Probability: {result_legit['fraud_probability']}")
print(f"Is Fraud:          {result_legit['is_fraud']}")
print(f"Risk Level:        {result_legit['risk_level']}")
print(f"\nTop Fraud Signals:")
for sig in result_legit['explanation']['top_fraud_signals']:
    print(f"  {sig['feature']}: "
          f"value={sig['value']:.3f}, "
          f"impact=+{sig['impact']:.4f}")
print(f"\nTop Legit Signals:")
for sig in result_legit['explanation']['top_legit_signals']:
    print(f"  {sig['feature']}: "
          f"value={sig['value']:.3f}, "
          f"impact={sig['impact']:.4f}")

=== TEST 2: LEGIT TRANSACTION ===

Fraud Probability: 0.0
Is Fraud:          False
Risk Level:        LOW

Top Fraud Signals:
  V8: value=-0.614, impact=+0.3740
  V16: value=-0.741, impact=+0.0441

Top Legit Signals:
  V4: value=-1.328, impact=-3.7463
  V14: value=0.266, impact=-1.8067
  V10: value=0.651, impact=-1.1578


In [21]:
# Cell 7 — Test 3: Speed test
import time

print("=== TEST 3: SPEED TEST ===\n")
print("Testing prediction speed (critical for production)")

times = []
for i in range(10):
    start = time.time()
    result = predict_fraud(fraud_raw,
                           return_explanation=True)
    end = time.time()
    times.append((end - start) * 1000)

avg_time = np.mean(times)
print(f"Average prediction time: {avg_time:.2f} ms")
print(f"Min time: {min(times):.2f} ms")
print(f"Max time: {max(times):.2f} ms")

if avg_time < 100:
    print("\n✅ Fast enough for production!")
    print("  Real-time fraud detection requires < 100ms")
elif avg_time < 500:
    print("\n⚠️ Acceptable but could be faster")
    print("  Consider caching SHAP explainer")
else:
    print("\n❌ Too slow for real-time")
    print("  Skip SHAP in production, use async")

=== TEST 3: SPEED TEST ===

Testing prediction speed (critical for production)
Average prediction time: 28.27 ms
Min time: 25.89 ms
Max time: 30.56 ms

✅ Fast enough for production!
  Real-time fraud detection requires < 100ms


In [22]:
# Cell 8 — Package everything together
pipeline = {
    # Core components
    'model': model,
    'explainer': explainer,
    'scaler_amount': scaler_amount,
    'scaler_hour': scaler_hour,

    # Configuration
    'threshold': threshold_config['final_threshold'],
    'feature_columns': X_test.columns.tolist(),

    # Functions
    'preprocess': preprocess_transaction,
    'predict': predict_fraud,

    # Metadata
    'metadata': {
        'model_type': 'XGBoost',
        'training_data': 'Original (no SMOTE)',
        'imbalance_handling': 'scale_pos_weight',
        'auc_roc': round(float(auc), 4),
        'f1_score': round(float(f1), 4),
        'threshold': threshold_config['final_threshold'],
        'n_features': model.n_features_in_,
        'version': '1.0.0',
        'created': pd.Timestamp.now().strftime(
            '%Y-%m-%d'
        )
    }
}

# Save complete pipeline
with open('../models/fraud_detection_pipeline.pkl',
          'wb') as f:
    pickle.dump(pipeline, f)

# Save metadata as JSON (human readable)
with open('../models/pipeline_metadata.json', 'w') as f:
    json.dump(pipeline['metadata'], f, indent=2)

print("✅ Complete pipeline saved!")
print("\n=== PIPELINE CONTENTS ===")
for key, val in pipeline.items():
    print(f"  {key:<20}: {type(val).__name__}")

print(f"\n=== PIPELINE METADATA ===")
for key, val in pipeline['metadata'].items():
    print(f"  {key:<20}: {val}")

✅ Complete pipeline saved!

=== PIPELINE CONTENTS ===
  model               : XGBClassifier
  explainer           : TreeExplainer
  scaler_amount       : StandardScaler
  scaler_hour         : StandardScaler
  threshold           : float
  feature_columns     : list
  preprocess          : function
  predict             : function
  metadata            : dict

=== PIPELINE METADATA ===
  model_type          : XGBoost
  training_data       : Original (no SMOTE)
  imbalance_handling  : scale_pos_weight
  auc_roc             : 0.9817
  f1_score            : 0.8454
  threshold           : 0.5
  n_features          : 32
  version             : 1.0.0
  created             : 2026-04-28


In [23]:
# Cell 9 — Load and verify saved pipeline
print("Verifying saved pipeline...\n")

with open('../models/fraud_detection_pipeline.pkl',
          'rb') as f:
    loaded_pipeline = pickle.load(f)

# Test loaded pipeline works
test_result = loaded_pipeline['predict'](
    fraud_raw,
    return_explanation=True
)

print("=== LOADED PIPELINE VERIFICATION ===\n")
print(f"✅ Pipeline loaded successfully!")
print(f"✅ Model type: "
      f"{loaded_pipeline['metadata']['model_type']}")
print(f"✅ Version: "
      f"{loaded_pipeline['metadata']['version']}")
print(f"✅ AUC-ROC: "
      f"{loaded_pipeline['metadata']['auc_roc']}")
print(f"✅ Threshold: "
      f"{loaded_pipeline['metadata']['threshold']}")
print(f"\n✅ Test prediction works!")
print(f"   Result: {test_result['is_fraud']} "
      f"(prob={test_result['fraud_probability']})")

Verifying saved pipeline...

=== LOADED PIPELINE VERIFICATION ===

✅ Pipeline loaded successfully!
✅ Model type: XGBoost
✅ Version: 1.0.0
✅ AUC-ROC: 0.9817
✅ Threshold: 0.5

✅ Test prediction works!
   Result: True (prob=0.9999)


In [24]:
# Cell 10 — Final folder structure check
print("=== FINAL MODELS FOLDER ===\n")

models_dir = '../models'
for file in sorted(os.listdir(models_dir)):
    filepath = os.path.join(models_dir, file)
    size = os.path.getsize(filepath) / 1024
    print(f"✅ {file:<45} {size:>8.1f} KB")

print(f"\n=== FINAL DATA/SPLITS FOLDER ===\n")
splits_dir = '../data/splits'
for file in sorted(os.listdir(splits_dir)):
    filepath = os.path.join(splits_dir, file)
    size = os.path.getsize(filepath) / 1024
    print(f"✅ {file:<45} {size:>8.1f} KB")

=== FINAL MODELS FOLDER ===

✅ baseline_results.json                              0.9 KB
✅ explain_transaction.pkl                            0.0 KB
✅ fraud_detection_pipeline.pkl                    3520.4 KB
✅ pipeline_metadata.json                             0.3 KB
✅ scaler_amount.pkl                                  0.6 KB
✅ scaler_hour.pkl                                    0.6 KB
✅ shap_explainer.pkl                              2693.0 KB
✅ shap_values_sample.npy                           125.1 KB
✅ threshold_config.json                              0.2 KB
✅ xgb_fraud_model.pkl                              826.1 KB
✅ xgb_results.json                                   1.4 KB

=== FINAL DATA/SPLITS FOLDER ===

✅ X_test.csv                                     32073.9 KB
✅ X_train.csv                                   128299.5 KB
✅ X_train_smote.csv                             264773.0 KB
✅ y_test.csv                                       166.9 KB
✅ y_train.csv                       